# SignalObjExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/SignalObjExamples.md`


Notebook source link: [SignalObjExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/SignalObjExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "SignalObjExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"SignalObjExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"SignalObjExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"SignalObjExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"SignalObjExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "sampleRate=100; t=0:1/sampleRate:10; freq=2;",
    "v1=sin(2*pi*freq*t); v2=sin(v1.^2); v=[v1;v2];",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "s1=SignalObj(t,v1,'Voltage','time','s','V',{'v1'});",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s1.plot;",
    "subplot(2,1,1); s.setMask({'v1'}); s.plot; s.resetMask;",
    "subplot(2,1,2); s.setMask({'v2'}); s.plot; size(s.dataToMatrix)",
    "s.resetMask;",
    "s=SignalObj(t,[v1; v1; v2] ,'Voltage','time','s','V',{'v1','v1','v2'});",
    "s.getSubSignal({'v1'}); %returns a SignalObj with both realizations of v1",
    "figure",
    "s.getSubSignal({'v1'}).plot;",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s.setXlabel('distance'); s.setXUnits('cm'); s.plot;",
    "figure",
    "subplot(2,1,1); s.setDataLabels({'r1','r2'}); s.setYLabel('Temperature'); s.setYUnits('C'); s.plot;",
    "subplot(2,1,2); s.setMaxTime(14); s.setMinTime(-2); s.plot;",
    "s.setName('testName'); %should work since we are using a method",
    "if(strcmp(s.name,'testName'))",
    "fprintf('Name successfully set \\n');",
    "else",
    "fprintf('Could not set name \\n');",
    "end",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "subplot(2,1,1); s.plot('v1',{{' ''k'' '}});",
    "subplot(2,1,2); s.plot('all',{{' ''k'' '},{' ''-.g'' '}});",
    "figure",
    "subplot(2,1,1); s.plot({'v1','v2'});",
    "subplot(2,1,2); s.plot({'v1','v2'},{{' ''k'' '},{' ''-.g'' '}});",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "s1=s.resample(.1*sampleRate);",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s1.plot;",
    "figure",
    "subplot(2,1,1); s.getSigInTimeWindow(-2,3).plot;",
    "subplot(2,1,2); s.plot;",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "figure",
    "s2=mean(s); %mean of each dimension;",
    "s5=s-s2;  %zero mean version of s;",
    "s5.plot;",
    "figure",
    "s2=mean(s,2); %mean of s across its dimensions;",
    "s2.plot;",
    "figure",
    "s4=2*s+s5;",
    "s4.plot;",
    "figure",
    "subplot(3,1,1);",
    "s.integral.plot;",
    "subplot(3,1,2);",
    "s.derivative.plot;",
    "subplot(3,1,3);",
    "s6=s.integral.derivative-s; %should equal zero;",
    "s6.plot;",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "figure;",
    "s.MTMspectrum;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for SignalObjExamples.")


In [ ]:
# SignalObjExamples: fixture-backed SignalObj parity checks.
from pathlib import Path
import nstat
from scipy.io import loadmat
from nstat.compat.matlab import SignalObj

m = loadmat(Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/SignalObjExamples_gold.mat", squeeze_me=True)
t = np.asarray(m["time_sig"], dtype=float).reshape(-1); v1 = np.asarray(m["v1_sig"], dtype=float).reshape(-1); v2 = np.asarray(m["v2_sig"], dtype=float).reshape(-1)
matlab_line("figure")
matlab_line("s.periodogram;")
matlab_line("sampleRate=5000; t=0:1/sampleRate:1; t=t'; freq=2;")
matlab_line("v1=sin(2*pi*freq*t); v2=sin(v1.^2);")
matlab_line("noise=.1*randn(length(t),6);")
matlab_line("data= [v1 v2 v2 v1 v2 v1] + noise;")
matlab_line("s=SignalObj(t,data,'Voltage','time','s','V',{'v1','v2','v2','v1','v1','v2'});")
matlab_line("figure;")
matlab_line("subplot(2,1,1); s.plot;")
matlab_line("subplot(2,1,2); s.plotAllVariability;")
matlab_line("s.plotVariability;")
matlab_line("figure;")
matlab_line("subplot(3,1,1); s.plotAllVariability('b');")
matlab_line("subplot(3,1,2); s.plotAllVariability('g',2);")
matlab_line("subplot(3,1,3); s.plotAllVariability('c',3,2,1);")
matlab_line("parity = struct();")
matlab_line("parity.sample_rate_hz = sampleRate;")
s = SignalObj(time=t, data=np.column_stack([v1, v2]), name="Voltage", units="V").setDataLabels(["v1", "v2"]).setXlabel("time").setXUnits("s").setYLabel("Voltage").setYUnits("V")
s.setMask(["v1"]); masked_cols = float(len(s.findIndFromDataMask())); s.resetMask()
s_resampled = s.resample(float(np.asarray(m["resample_hz_sig"]).reshape(-1)[0])); s_win = s.getSigInTimeWindow(float(np.asarray(m["window_t0_sig"]).reshape(-1)[0]), float(np.asarray(m["window_t1_sig"]).reshape(-1)[0]))
f_per, p_per = s.periodogram(); expected_peak = int(np.asarray(m["periodogram_peak_idx_sig"], dtype=int).reshape(-1)[0]); peak_idx = int(np.argmax(p_per))
s.setName("testName")
s_der = s.derivative()
s_int = s.integral()
s_sub = s.getSubSignal([0])
s_repeat = SignalObj(time=t, data=np.column_stack([v1, v1, v2]), name="Voltage", units="V").setDataLabels(["v1", "v1", "v2"])
s_repeat_v1 = s_repeat.getSubSignal([0, 1])

fig, ax = plt.subplots(2, 2, figsize=(10, 6))
plt.sca(ax[0, 0]); s.plot(); ax[0, 0].set_title("SignalObj.plot")
plt.sca(ax[0, 1]); s_resampled.plot(); ax[0, 1].set_title("resample")
plt.sca(ax[1, 0]); s_win.plot(); ax[1, 0].set_title("time window")
ax[1, 1].plot(f_per, p_per, "k", linewidth=1.0); ax[1, 1].set_title("periodogram")
plt.tight_layout(); plt.show()

assert masked_cols == float(np.asarray(m["masked_cols_sig"]).reshape(-1)[0])
assert peak_idx == expected_peak
assert s.getNumSamples() == int(np.asarray(m["n_samples_sig"], dtype=int).reshape(-1)[0])
assert s_resampled.getNumSamples() == int(np.asarray(m["resampled_n_samples_sig"], dtype=int).reshape(-1)[0])
assert s_win.getNumSamples() == int(np.asarray(m["window_n_samples_sig"], dtype=int).reshape(-1)[0])
assert s_der.getNumSamples() == s.getNumSamples()
assert s_int.shape[0] == s.getNumSamples()
assert s_sub.getNumSignals() == 1
assert s_repeat_v1.getNumSignals() == 2

CHECKPOINT_METRICS = {
    "masked_cols": float(masked_cols),
    "periodogram_peak_idx": float(peak_idx),
    "resampled_samples": float(s_resampled.getNumSamples()),
    "window_samples": float(s_win.getNumSamples()),
}
CHECKPOINT_LIMITS = {
    "masked_cols": (1.0, 1.0),
    "periodogram_peak_idx": (0.0, 50000.0),
    "resampled_samples": (10.0, 2000.0),
    "window_samples": (10.0, 5000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
